# Robust estimator comparison

This notebook compares frozen estimator audit outputs. It does not tune or propose thresholds. Calibration and held-out evaluation are kept separate, and seed-level displays use the simulation job/seed as the independent unit.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "scgeo_revision_matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
})

SPLIT_COLORS = {"calibration": "#4C78A8", "evaluation": "#F58518"}
METHOD_ORDER = ["shift_mean", "robust_mean", "robust_median", "robust_trimmed_mean", "robust_geometric_median"]
METHOD_LABELS = {
    "shift_mean": "Mean",
    "robust_mean": "Robust mean",
    "robust_median": "Median",
    "robust_trimmed_mean": "Trimmed mean",
    "robust_geometric_median": "Geometric median",
}


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "configs" / "manuscript_benchmark_v1.json").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory")


REPO_ROOT = find_repo_root()
CONFIG_PATH = REPO_ROOT / "configs" / "manuscript_benchmark_v1.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))


def resolve_config_path(value: str) -> Path:
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()


BENCHMARK_DIR = resolve_config_path(os.environ.get(CONFIG["benchmark_dir_env"], CONFIG["default_benchmark_dir"]))
OUTPUT_DIR = resolve_config_path(os.environ.get(CONFIG["notebook_output_env"], CONFIG["default_output_dir"]))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def rel_display(path: Path) -> str:
    path = path.resolve()
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not-installed"


def git_commit(path: Path) -> str | None:
    if not (path / ".git").exists():
        return None
    try:
        return subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_manifest() -> pd.DataFrame:
    manifest = pd.read_csv(REPO_ROOT / CONFIG["results_manifest"])
    required_columns = {"relative_path", "role", "size_bytes", "sha256", "protocol_version", "profile", "source_commit"}
    missing = required_columns.difference(manifest.columns)
    if missing:
        raise AssertionError(f"Manifest is missing columns: {sorted(missing)}")
    return manifest


def load_checksum_manifest() -> dict[str, str]:
    checksums = {}
    path = REPO_ROOT / CONFIG["checksums_manifest"]
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        sha, rel = line.split("  ", 1)
        checksums[rel] = sha
    return checksums


def verify_inputs(required_paths: list[str] | None = None, exhaustive_checksums: bool = False) -> dict[str, object]:
    if not BENCHMARK_DIR.exists():
        raise FileNotFoundError(f"Frozen benchmark directory not found: {BENCHMARK_DIR}")

    manifest = load_manifest()
    checksum_manifest = load_checksum_manifest()
    manifest_idx = manifest.set_index("relative_path", drop=False)
    if set(checksum_manifest) != set(manifest_idx.index):
        raise AssertionError("benchmark_files.csv and checksums.sha256 do not describe the same files")
    for rel, expected in checksum_manifest.items():
        actual = manifest_idx.loc[rel, "sha256"]
        if actual != expected:
            raise AssertionError(f"Checksum manifest disagrees with benchmark_files.csv for {rel}")

    if set(manifest["protocol_version"].unique()) != {CONFIG["protocol_version"]}:
        raise AssertionError("Manifest protocol_version does not match config")
    if set(manifest["profile"].unique()) != {CONFIG["profile"]}:
        raise AssertionError("Manifest profile does not match config")
    if set(manifest["source_commit"].unique()) != {CONFIG["expected_source_commit"]}:
        raise AssertionError("Manifest source_commit does not match config")

    selected = set(required_paths or [])
    selected.add("manuscript_jobs.csv")
    selected.update(manifest.loc[manifest["role"].isin(["job_config", "job_status"]), "relative_path"].tolist())
    if exhaustive_checksums:
        selected = set(manifest_idx.index)

    unknown = sorted(selected.difference(manifest_idx.index))
    if unknown:
        raise AssertionError(f"Requested files are not present in the manifest: {unknown[:10]}")

    missing_files = []
    checksum_mismatches = []
    for rel in sorted(selected):
        path = BENCHMARK_DIR / rel
        if not path.exists():
            missing_files.append(rel)
            continue
        expected = manifest_idx.loc[rel, "sha256"]
        actual = sha256_file(path)
        if actual != expected:
            checksum_mismatches.append((rel, expected, actual))
    if missing_files:
        raise FileNotFoundError(f"Missing frozen benchmark files: {missing_files[:10]}")
    if checksum_mismatches:
        first = checksum_mismatches[0]
        raise AssertionError(f"Checksum mismatch for {first[0]}: expected {first[1]}, observed {first[2]}")

    jobs = pd.read_csv(BENCHMARK_DIR / "manuscript_jobs.csv")
    if len(jobs) != CONFIG["expected_jobs"]["total"]:
        raise AssertionError(f"Expected {CONFIG['expected_jobs']['total']} jobs, observed {len(jobs)}")
    if int((jobs["status"] == "completed").sum()) != CONFIG["expected_jobs"]["completed"]:
        raise AssertionError("Not all frozen manuscript jobs are marked completed")

    config_rows = []
    for rel in sorted(manifest.loc[manifest["role"] == "job_config", "relative_path"]):
        data = json.loads((BENCHMARK_DIR / rel).read_text(encoding="utf-8"))
        config_rows.append({
            "job_id": data["job_id"],
            "profile": data["profile"],
            "scenario": data["scenario"],
            "seed_split": data["seed_split"],
            "seed": int(data["seed"]),
            "output_dir": data.get("output_dir", ""),
            "n_boot": data.get("profile_cfg", {}).get("n_boot"),
        })
    job_configs = pd.DataFrame(config_rows)
    if len(job_configs) != CONFIG["expected_jobs"]["total"]:
        raise AssertionError("Job config count does not match expected total")
    if set(job_configs["profile"]) != {CONFIG["profile"]}:
        raise AssertionError("Per-job profile does not match the manuscript profile")
    if sorted(job_configs["scenario"].unique()) != CONFIG["expected_scenarios"]:
        raise AssertionError("Per-job scenarios do not match the revision config")
    if not job_configs["output_dir"].astype(str).str.endswith(CONFIG["protocol_version"]).all():
        raise AssertionError("Per-job output_dir does not record the expected protocol version")

    split_counts = job_configs["seed_split"].value_counts().to_dict()
    for split, expected_count in [("calibration", CONFIG["expected_jobs"]["calibration"]), ("evaluation", CONFIG["expected_jobs"]["evaluation"] )]:
        if int(split_counts.get(split, 0)) != expected_count:
            raise AssertionError(f"Unexpected {split} job count: {split_counts.get(split, 0)}")
    seeds_by_split = job_configs.groupby("seed_split")["seed"].apply(lambda values: sorted(set(map(int, values)))).to_dict()
    for split, expected_seeds in CONFIG["expected_seed_splits"].items():
        if seeds_by_split.get(split) != expected_seeds:
            raise AssertionError(f"Unexpected seeds for {split}: {seeds_by_split.get(split)}")

    source_repo = resolve_config_path(os.environ.get(CONFIG["source_repository_env"], CONFIG["default_source_repository"]))
    source_repo_commit = git_commit(source_repo)
    if source_repo_commit is not None and source_repo_commit != CONFIG["expected_source_commit"]:
        raise AssertionError(f"Source repository commit mismatch: {source_repo_commit}")

    return {
        "manifest": manifest,
        "jobs": jobs,
        "job_configs": job_configs,
        "checksum_files_verified": len(selected),
        "source_repo_commit": source_repo_commit,
        "source_repo_commit_verified": source_repo_commit == CONFIG["expected_source_commit"],
        "benchmark_dir": BENCHMARK_DIR,
        "output_dir": OUTPUT_DIR,
    }


def read_audit(filename: str) -> pd.DataFrame:
    return pd.read_csv(BENCHMARK_DIR / "audit" / filename)


def write_table_artifact(stem: str, table: pd.DataFrame, alt_text: str | None = None) -> pd.DataFrame:
    source_dir = OUTPUT_DIR / "figure_sources"
    source_dir.mkdir(parents=True, exist_ok=True)
    table_path = source_dir / f"{stem}.csv"
    table.to_csv(table_path, index=False)
    rows = [{"artifact": stem, "source_csv": rel_display(table_path)}]
    if alt_text is not None:
        alt_dir = OUTPUT_DIR / "alt_text"
        alt_dir.mkdir(parents=True, exist_ok=True)
        alt_path = alt_dir / f"{stem}.txt"
        alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
        rows[0]["alt_text"] = rel_display(alt_path)
    return pd.DataFrame(rows)


def write_figure_bundle(stem: str, fig: plt.Figure, source_table: pd.DataFrame, alt_text: str) -> pd.DataFrame:
    fig_dir = OUTPUT_DIR / "figures"
    source_dir = OUTPUT_DIR / "figure_sources"
    alt_dir = OUTPUT_DIR / "alt_text"
    for directory in (fig_dir, source_dir, alt_dir):
        directory.mkdir(parents=True, exist_ok=True)
    png_path = fig_dir / f"{stem}.png"
    svg_path = fig_dir / f"{stem}.svg"
    csv_path = source_dir / f"{stem}.csv"
    alt_path = alt_dir / f"{stem}.txt"
    source_table.to_csv(csv_path, index=False)
    fig.savefig(svg_path, format="svg", bbox_inches="tight", metadata={"Date": None})
    fig.savefig(png_path, format="png", bbox_inches="tight", metadata={"Software": "matplotlib"})
    alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
    plt.close(fig)
    return pd.DataFrame([{ 
        "figure": stem,
        "png": rel_display(png_path),
        "svg": rel_display(svg_path),
        "source_csv": rel_display(csv_path),
        "alt_text": rel_display(alt_path),
    }])


def write_metadata(stem: str, extra: dict[str, object] | None = None) -> pd.DataFrame:
    metadata_dir = OUTPUT_DIR / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    source_repo = resolve_config_path(os.environ.get(CONFIG["source_repository_env"], CONFIG["default_source_repository"]))
    metadata = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "packages": {name: package_version(name) for name in ["scgeo", "pandas", "numpy", "matplotlib", "nbformat", "nbclient"]},
        "notebook_repository_commit": git_commit(REPO_ROOT),
        "source_commit_expected": CONFIG["expected_source_commit"],
        "source_repository_commit": git_commit(source_repo),
        "protocol_version": CONFIG["protocol_version"],
        "profile": CONFIG["profile"],
        "benchmark_dir": str(BENCHMARK_DIR),
        "output_dir": str(OUTPUT_DIR),
        "independent_unit": CONFIG["independent_unit"],
        "threshold_policy": CONFIG["threshold_policy"],
    }
    if extra:
        metadata.update(extra)
    path = metadata_dir / f"{stem}_metadata.json"
    path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return pd.DataFrame([{"metadata": rel_display(path)}])


def split_source_note() -> pd.DataFrame:
    return pd.DataFrame([{
        "calibration_seeds": ",".join(map(str, CONFIG["expected_seed_splits"]["calibration"])),
        "evaluation_seeds": ",".join(map(str, CONFIG["expected_seed_splits"]["evaluation"])),
        "independent_unit": CONFIG["independent_unit"],
        "threshold_policy": CONFIG["threshold_policy"],
    }])

In [ ]:
required = [
    "audit/estimator_comparison.csv",
    "audit/shift_detection_by_seed.csv",
    "audit/null_false_call_by_seed.csv",
    "audit/bootstrap_uncertainty_by_seed.csv",
    "audit/limitations.csv",
]
context = verify_inputs(required_paths=required, exhaustive_checksums=False)
pd.DataFrame([{
    "protocol": CONFIG["protocol_version"],
    "source_commit": CONFIG["expected_source_commit"],
    "checksum_files_verified": context["checksum_files_verified"],
    "independent_unit": CONFIG["independent_unit"],
}])

In [ ]:
estimator = read_audit("estimator_comparison.csv")
outlier = estimator[(estimator["scenario"] == "outlier_contamination") & (estimator["metric"] == "magnitude_error_mean")].copy()
outlier["method_label"] = outlier["method"].map(METHOD_LABELS)
outlier["method"] = pd.Categorical(outlier["method"], categories=METHOD_ORDER, ordered=True)
outlier = outlier.sort_values(["seed_split", "method"])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True, constrained_layout=True)
for ax, split in zip(axes, ["calibration", "evaluation"]):
    subset = outlier[outlier["seed_split"] == split].copy()
    x = np.arange(len(subset))
    ax.bar(x, subset["mean"], color=SPLIT_COLORS[split], width=0.65)
    yerr = np.vstack([(subset["mean"] - subset["ci95_low"]).clip(lower=0), (subset["ci95_high"] - subset["mean"]).clip(lower=0)])
    ax.errorbar(x, subset["mean"], yerr=yerr, fmt="none", color="#222222", capsize=3, linewidth=1)
    ax.set_xticks(x, subset["method_label"], rotation=35, ha="right")
    ax.set_title(split.capitalize())
    ax.set_ylabel("Mean magnitude error")
    ax.set_ylim(bottom=0)

eval_rows = outlier[outlier["seed_split"] == "evaluation"].set_index("method")
best_method = eval_rows["mean"].idxmin()
alt = (
    f"Outlier-contamination magnitude error by estimator with calibration and held-out evaluation in separate panels. "
    f"Held-out mean magnitude error is lowest for {METHOD_LABELS[best_method]} at {eval_rows.loc[best_method, 'mean']:.3f}; "
    "classification was otherwise saturated in this scenario."
)
artifacts_outlier = write_figure_bundle("01_outlier_magnitude_error", fig, outlier, alt)
artifacts_outlier

In [ ]:
null_calls = read_audit("null_false_call_by_seed.csv")
balanced = null_calls[(null_calls["scenario"] == "balanced_replicate_heterogeneity") & (null_calls["seed_split"] == "evaluation")].copy()
balanced["method"] = pd.Categorical(balanced["method"], categories=METHOD_ORDER, ordered=True)
balanced = balanced.sort_values(["method", "seed"])
heat = balanced.pivot(index="method", columns="seed", values="false_call_rate").reindex(METHOD_ORDER)

fig, ax = plt.subplots(figsize=(9.4, 3.8), constrained_layout=True)
im = ax.imshow(heat.values, vmin=0, vmax=1, aspect="auto", cmap="YlOrRd")
ax.set_yticks(np.arange(len(METHOD_ORDER)), [METHOD_LABELS[m] for m in METHOD_ORDER])
ax.set_xticks(np.arange(len(heat.columns)), heat.columns.astype(str))
ax.set_xlabel("Held-out seed")
ax.set_title("Balanced-replicate neutral false-call rate by seed")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iloc[i, j]
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=7, color="#222222")
fig.colorbar(im, ax=ax, label="False-call rate")

balanced_summary = (
    balanced.groupby("method", observed=True)
    .agg(n_jobs=("job_id", "nunique"), jobs_with_any_false_call=("any_false_call", "sum"), mean_false_call_rate=("false_call_rate", "mean"))
    .reset_index()
)
balanced_summary["method_label"] = balanced_summary["method"].map(METHOD_LABELS)
median_row = balanced_summary[balanced_summary["method"] == "robust_median"].iloc[0]
geomed_row = balanced_summary[balanced_summary["method"] == "robust_geometric_median"].iloc[0]
alt = (
    f"Balanced-replicate held-out false-call heatmap by seed. Robust median has false calls in "
    f"{int(median_row['jobs_with_any_false_call'])} of {int(median_row['n_jobs'])} jobs; geometric median has false calls in "
    f"{int(geomed_row['jobs_with_any_false_call'])} of {int(geomed_row['n_jobs'])} jobs; mean, robust mean, and trimmed mean have zero jobs with false calls."
)
artifacts_balanced = write_figure_bundle("01_balanced_replicate_seed_dependence", fig, balanced, alt)
write_table_artifact("01_balanced_replicate_seed_summary", balanced_summary)
artifacts_balanced

In [ ]:
bootstrap = read_audit("bootstrap_uncertainty_by_seed.csv")
assessed = bootstrap[bootstrap["n_assessed"].fillna(0) > 0].copy()
assessed = assessed.sort_values(["seed_split", "scenario", "seed"])

fig, ax = plt.subplots(figsize=(7.2, 3.8), constrained_layout=True)
plot_data = [assessed.loc[assessed["seed_split"] == split, "coverage_rate_assessed"].dropna().values for split in ["calibration", "evaluation"]]
box = ax.boxplot(plot_data, labels=["Calibration", "Held-out"], patch_artist=True, widths=0.55)
for patch, split in zip(box["boxes"], ["calibration", "evaluation"]):
    patch.set_facecolor(SPLIT_COLORS[split])
    patch.set_alpha(0.78)
for i, split in enumerate(["calibration", "evaluation"], start=1):
    values = assessed.loc[assessed["seed_split"] == split, "coverage_rate_assessed"].dropna().values
    ax.scatter(np.full_like(values, i, dtype=float), values, color="#222222", s=12, alpha=0.55, zorder=3)
ax.set_ylim(-0.04, 1.04)
ax.set_ylabel("Assessed-state coverage rate")
ax.set_title("Bootstrap uncertainty coverage by independent job")

coverage_summary = (
    assessed.groupby("seed_split", as_index=False)
    .agg(n_jobs=("job_id", "nunique"), mean_coverage=("coverage_rate_assessed", "mean"), median_coverage=("coverage_rate_assessed", "median"))
)
eval_cov = coverage_summary[coverage_summary["seed_split"] == "evaluation"].iloc[0]
alt = (
    f"Bootstrap assessed-state coverage rates are shown by independent job with calibration and held-out evaluation separated. "
    f"Held-out mean coverage is {eval_cov['mean_coverage']:.3f} across {int(eval_cov['n_jobs'])} jobs, so uncertainty coverage is imperfect under the frozen profile."
)
artifacts_bootstrap = write_figure_bundle("01_bootstrap_uncertainty_coverage", fig, assessed, alt)
write_table_artifact("01_bootstrap_coverage_summary", coverage_summary)
artifacts_bootstrap

In [ ]:
negative_report = pd.concat([
    balanced_summary.assign(negative_result="balanced-replicate seed dependence"),
    coverage_summary.assign(negative_result="imperfect bootstrap uncertainty coverage"),
], ignore_index=True, sort=False)
write_table_artifact("01_negative_results", negative_report)
metadata = write_metadata("01_robust_estimator_comparison", {
    "checksum_files_verified": context["checksum_files_verified"],
    "figures": [
        "01_outlier_magnitude_error",
        "01_balanced_replicate_seed_dependence",
        "01_bootstrap_uncertainty_coverage",
    ],
})
metadata